# Análise Exploratória dos Dados de Triagem de Autismo em Crianças Pequenas

Este notebook tem finalidade exclusivamente acadêmica e não deve ser utilizado para diagnóstico ou orientação clínica.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
project_dir = Path("..") if Path.cwd().name == "notebooks" else Path(".")
data_dir = project_dir / "data" / "raw"
dataset_path = data_dir / "Toddler Autism dataset July 2018.csv"
archive_path = data_dir / "autism-screening-for-toddlers.zip"
dataset_url = "https://www.kaggle.com/api/v1/datasets/download/fabdelja/autism-screening-for-toddlers"

if not dataset_path.exists():
    data_dir.mkdir(parents=True, exist_ok=True)
    urlretrieve(dataset_url, archive_path)
    with ZipFile(archive_path) as archive:
        archive.extractall(data_dir)

In [3]:
df = pd.read_csv(dataset_path)
df.head()

,Case_No,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,Age_Mons,Qchat-10-Score,Sex,Ethnicity,Jaundice,Family_mem_with_ASD,Who completed the test,Class/ASD Traits
0,1,0,0,0,0,0,0,1,1,0,1,28,3,f,middle eastern,yes,no,family member,No
1,2,1,1,0,0,0,1,1,0,0,0,36,4,m,White European,yes,no,family member,Yes
2,3,1,0,0,0,0,0,1,1,0,1,36,4,m,middle eastern,yes,no,family member,Yes
3,4,1,1,1,1,1,1,1,1,1,1,24,10,m,Hispanic,no,no,family member,Yes
4,5,1,1,0,1,1,1,1,1,1,1,20,9,f,White European,no,yes,family member,Yes


## Estrutura inicial do dataset

In [4]:
print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")

Linhas: 1054
Colunas: 19


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1054 entries, 0 to 1053
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Case_No                 1054 non-null   int64
 1   A1                      1054 non-null   int64
 2   A2                      1054 non-null   int64
 3   A3                      1054 non-null   int64
 4   A4                      1054 non-null   int64
 5   A5                      1054 non-null   int64
 6   A6                      1054 non-null   int64
 7   A7                      1054 non-null   int64
 8   A8                      1054 non-null   int64
 9   A9                      1054 non-null   int64
 10  A10                     1054 non-null   int64
 11  Age_Mons                1054 non-null   int64
 12  Qchat-10-Score          1054 non-null   int64
 13  Sex                     1054 non-null   str  
 14  Ethnicity               1054 non-null   str  
 15  Jaundice                1054 non

## Valores ausentes

In [6]:
df.isna().sum().to_frame("valores_ausentes")

,valores_ausentes
Case_No,0
A1,0
A2,0
A3,0
A4,0
A5,0
A6,0
A7,0
A8,0
A9,0


## Limpeza dos dados

Padronização dos nomes das colunas (a coluna `Class/ASD Traits ` possui um espaço extra no final) e das categorias registradas com grafias inconsistentes, além da verificação de registros duplicados.

In [7]:
df.columns = df.columns.str.strip()

print(f"Registros duplicados: {df.duplicated().sum()}")
df.columns.tolist()

Registros duplicados: 0


['Case_No',
 'A1',
 'A2',
 'A3',
 'A4',
 'A5',
 'A6',
 'A7',
 'A8',
 'A9',
 'A10',
 'Age_Mons',
 'Qchat-10-Score',
 'Sex',
 'Ethnicity',
 'Jaundice',
 'Family_mem_with_ASD',
 'Who completed the test',
 'Class/ASD Traits']

In [8]:
for column in ["Ethnicity", "Who completed the test"]:
    df[column] = df[column].str.strip().str.title()

print(sorted(df["Ethnicity"].unique()))
print(sorted(df["Who completed the test"].unique()))

['Asian', 'Black', 'Hispanic', 'Latino', 'Middle Eastern', 'Mixed', 'Native Indian', 'Others', 'Pacifica', 'South Asian', 'White European']
['Family Member', 'Health Care Professional', 'Others', 'Self']


## Ajuste de tipos

As colunas de texto representam categorias com poucos valores distintos e são convertidas para o tipo `category`. As colunas `A1` a `A10`, `Age_Mons` e `Qchat-10-Score` permanecem numéricas.

In [9]:
categorical_columns = [
    "Sex",
    "Ethnicity",
    "Jaundice",
    "Family_mem_with_ASD",
    "Who completed the test",
    "Class/ASD Traits",
]
df[categorical_columns] = df[categorical_columns].astype("category")
df.dtypes

Case_No                      int64
A1                           int64
A2                           int64
A3                           int64
A4                           int64
A5                           int64
A6                           int64
A7                           int64
A8                           int64
A9                           int64
A10                          int64
Age_Mons                     int64
Qchat-10-Score               int64
Sex                       category
Ethnicity                 category
Jaundice                  category
Family_mem_with_ASD       category
Who completed the test    category
Class/ASD Traits          category
dtype: object

## Tratamento de valores ausentes

A verificação anterior mostrou que o dataset não possui valores ausentes, portanto nenhuma imputação é aplicada de fato. Ainda assim, o tratamento abaixo é mantido de forma preventiva: colunas numéricas seriam preenchidas com a mediana e colunas categóricas com a moda.

In [10]:
if df.isna().any().any():
    numeric_columns = df.select_dtypes("number").columns
    df[numeric_columns] = df[numeric_columns].fillna(df[numeric_columns].median())

    category_columns = df.select_dtypes("category").columns
    df[category_columns] = df[category_columns].fillna(df[category_columns].mode().iloc[0])

print(f"Valores ausentes após o tratamento: {df.isna().sum().sum()}")

Valores ausentes após o tratamento: 0


## Salvamento dos dados tratados

O dataset limpo é salvo em `data/processed/` para ser reutilizado nas próximas etapas da análise.

In [11]:
processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
processed_path = processed_dir / "toddler_autism_clean.csv"
df.to_csv(processed_path, index=False)
print(f"Arquivo salvo em: {processed_path}")

Arquivo salvo em: ../data/processed/toddler_autism_clean.csv
